# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {getattr(metadata, 'citeAs', None)}")
print(f"Version: {getattr(metadata, 'version', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record set @ids available in the metadata
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
elif hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = []

if isinstance(record_sets, dict):
    record_sets = [record_sets]

if len(record_sets) == 0:
    # If not present, let's try reading from the dataset directly
    record_set_ids = list(dataset._record_sets.keys())
else:
    record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]

print("Record Sets and their @id:")
for i, rs_id in enumerate(record_set_ids):
    print(f"  {i+1}. {rs_id}")

# For an initial overview, display first field IDs of the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nFields in the first record set ({first_rs}):")
    try:
        rs_obj = dataset._record_sets[first_rs]
        field_list = rs_obj.get('field', [])
        if isinstance(field_list, dict):
            field_list = [field_list]
        for i, field in enumerate(field_list):
            if isinstance(field, dict) and '@id' in field:
                fid = field['@id']
            else:
                fid = field
            print(f"  Field {i+1}: {fid}")
    except Exception as e:
        print("  (Could not retrieve fields directly, see next cell for DataFrame columns)")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all discovered record sets
dataframes = {}
for record_set in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows from {record_set}")
    except Exception as e:
        print(f"Could not load {record_set}: {e}")

# Choose the main record set (usually the largest or with most columns)
if len(dataframes) > 0:
    # Use the record set with the most columns
    main_record_set_id = max(dataframes.keys(), key=lambda k: dataframes[k].shape[1])
    print(f"\nPrimary record set for analysis: {main_record_set_id}")
    print("Columns available:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping. All fields are referenced by their `@id` as per the Croissant specification.

In [ ]:
# Identify a numeric field @id (by default, use "cr:abundance" or similar)
df = dataframes[main_record_set_id].copy()

print("Columns detected:\n", df.columns.tolist())
# Attempt to select common proteomics fields
candidates = [
    'cr:abundance',
    'cr:coverage',
    'cr:mw',
    'cr:peptide_count',
    'cr:score',
    'cr:MW',
    'cr:normalized_abundance',
    'cr:intensity',
]
numeric_field_id = None
for c in candidates:
    if c in df.columns:
        numeric_field_id = c
        break
if not numeric_field_id:
    # Fallback: auto-detect first numeric column
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
print(f"Using numeric field: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = df[df[numeric_field_id] > threshold].copy() if numeric_field_id else df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {len(filtered_df)} rows\n")

if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if present, e.g. cr:modification or cr:sample
group_field_candidates = [
    'cr:modification',
    'cr:sample',
    'cr:isoform',
    'cr:gene',
    'cr:experiment',
    'cr:label',
    'cr:accession',
]
group_field = None
for gf in group_field_candidates:
    if gf in df.columns and pd.api.types.is_object_dtype(df[gf]):
        group_field = gf
        break

if group_field and numeric_field_id:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
    print(f"Top groups by mean {numeric_field_id} for '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a selected numeric field, and/or explore relationships between categorical and numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of abundance or selected numeric field
plt.figure(figsize=(8, 5))
if numeric_field_id:
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group if available
if group_field and numeric_field_id:
    plt.figure(figsize=(10, 6))
    # Limit to top 10 groups by size
    top_groups = df[group_field].value_counts().index[:10]
    sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end dataset loading, inspection, and simple exploratory analysis using `mlcroissant`. The FAIR^2 dataset provides structured mass spectrometry data on extracellular vesicle proteins, with detailed information on each protein record. You may extend this analysis for domain-specific insights or advanced modeling tasks.